# NumPy — zaawansowane

**Programowanie w Pythonie II** | Wykład 4
**Politechnika Opolska** | Analityka danych w biznesie

---

## Co dzisiaj?

Na W03 poznaliśmy podstawy NumPy: tworzenie tablic, indeksowanie, operacje wektorowe, filtrowanie, axis. Teraz wchodzimy głębiej:

```mermaid
graph TD
    A["W03: NumPy podstawy"] --> B["Tworzenie tablic"]
    A --> C["Indeksowanie i slicing"]
    A --> D["Operacje wektorowe"]
    A --> E["axis=0 / axis=1"]
    F["W04: NumPy zaawansowane"] --> G["Broadcasting"]
    F --> H["Reshape i stacking"]
    F --> I["where, select, sort, unique"]
    F --> J["Generowanie danych"]
    F --> K["NaN, digitize, diff"]
    A -.-> F
    F -.-> L["W05: Pandas"]
```

### Po tym wykładzie potrafisz:

1. **Wyjaśnić** mechanizm broadcastingu i stosować go w obliczeniach (Bloom 2-3)
2. **Zmieniać** kształt tablic za pomocą reshape, flatten, stacking (Bloom 3)
3. **Stosować** zaawansowane operacje: where, clip, sort, argsort, unique, korelacja (Bloom 3)
4. **Generować** dane syntetyczne z rozkładów i obliczać statystyki opisowe (Bloom 3)
5. **Obsługiwać** brakujące dane (np.nan) i stosować wielowarunkowe operacje (Bloom 3)

In [ ]:
import numpy as np
print(f"NumPy wersja: {np.__version__}")

---

## 1. Broadcasting — operacje na tablicach różnych kształtów

Broadcasting to mechanizm, dzięki któremu NumPy potrafi operować na tablicach **różnych kształtów** — bez pętli. Już go używaliście na pierwszej części zjazdu, nie wiedząc o tym.

### Analogia do Excela

| Co robisz w Excelu | Co robi NumPy |
|---------------------|---------------|
| Kolumna cen × komórka z rabatem — kopiujesz formułę w dół | `ceny * 0.9` — skalar rozciąga się na całą tablicę |
| Tabela cen × wiersz rabatów per kwartał — kopiujesz w dół i w prawo | `ceny * (1 - rabat)` — wektor rozciąga się na wiersze |
| Tabela cen × kolumna premii per produkt — kopiujesz formułę | `ceny * premia` — kolumna rozciąga się na kolumny |

In [ ]:
# Najprostszy broadcasting: tablica × skalar
a = np.array([1, 2, 3, 4])
print(f"a = {a}")
print(f"a * 10 = {a * 10}")  # NumPy rozciąga 10 na [10, 10, 10, 10]
print(f"\na.shape = {a.shape}, skalar nie ma shape — NumPy rozciąga go automatycznie")

In [ ]:
# Sprawdzmy kształty — to klucz do zrozumienia broadcastingu
print(f"a.shape = {a.shape}")   # (4,) — wektor 4 elementów
print(f"Skalar 10 nie ma shape — NumPy rozciąga go automatycznie")
print()
print("Co się stało krok po kroku:")
print("  a:       [1, 2, 3, 4]       shape (4,)")
print("  10:      10                  shape ()  — skalar")
print("  NumPy rozciąga 10 na [10, 10, 10, 10] i mnoży element po elemencie")

In [ ]:
# Broadcasting wierszowy: macierz × wektor
# Macierz cen: 3 produkty × 4 kwartały
ceny = np.array([[100, 110, 120, 130],
                  [200, 210, 220, 230],
                  [50,  55,  60,  65]])

rabat = np.array([0.05, 0.10, 0.15, 0.20])  # inny rabat per kwartał

ceny_po_rabacie = ceny * (1 - rabat)
print(f"Ceny (3×4):\n{ceny}")
print(f"Rabat per kwartał: {rabat}")
print(f"Po rabacie:\n{ceny_po_rabacie}")
# Wektor 4-elementowy pasuje do 4 kolumn — rozciąga się na każdy wiersz

In [ ]:
# Dlaczego to zadziałało?
print(f"ceny.shape:  {ceny.shape}")    # (3, 4)
print(f"rabat.shape: {rabat.shape}")   # (4,)
print()
print("Porównanie od prawej:")
print("  ceny:  (3, 4)")
print("  rabat:    (4,)")
print("           4 == 4  — wymiary zgodne")
print("  NumPy rozciąga rabat na każdy wiersz (produkt)")

In [ ]:
# Broadcasting kolumnowy: macierz × wektor kolumnowy
premia = np.array([[1.2],    # produkt A: +20%
                    [1.0],    # produkt B: bez premii
                    [1.5]])   # produkt C: +50%

ceny_z_premia = ceny * premia
print(f"Premia per produkt: {premia.flatten()}")
print(f"Ceny z premią:\n{ceny_z_premia}")
# Kolumna (3,1) rozciąga się na 4 kolumny

In [ ]:
# Dlaczego potrzebujemy (3,1) a nie (3,)?
print(f"ceny.shape:   {ceny.shape}")     # (3, 4)
print(f"premia.shape: {premia.shape}")   # (3, 1)
print()
print("Porównanie od prawej:")
print("  ceny:   (3, 4)")
print("  premia: (3, 1)")
print("           4 vs 1 — 1 rozciąga się do 4")
print("           3 == 3")
print("  Każdy WIERSZ (produkt) mnożony przez swoją premię")

### Zasady broadcastingu

NumPy porównuje kształty **od prawej** do lewej. Wymiar musi być **taki sam** lub **równy 1**.

| Kształty | Wynik | Dlaczego |
|----------|-------|----------|
| `(3, 4)` × `(4,)` | `(3, 4)` | 4 == 4 |
| `(3, 4)` × `(3, 1)` | `(3, 4)` | 3 == 3, 1 rozciąga się do 4 |
| `(3, 4)` × `(1, 4)` | `(3, 4)` | 1 rozciąga się do 3, 4 == 4 |
| `(3, 4)` × `(3,)` | **BŁĄD!** | 4 ≠ 3 — nie pasuje |

**Najczęstszy błąd:** macierz 3×4 razy wektor 3 elementów. Rozwiązanie: `.reshape(3, 1)`.

In [ ]:
# Sprawdzenie zasad — co działa, co nie
m = np.ones((3, 4))

print(f"(3,4) * (4,)   → shape {(m * np.array([1, 2, 3, 4])).shape}  ✔  4==4")
print(f"(3,4) * (3,1)  → shape {(m * np.array([[1],[2],[3]])).shape}  ✔  1→4, 3==3")

try:
    m * np.array([1, 2, 3])    # (3,4) * (3,) — nie pasuje!
except ValueError as e:
    print(f"(3,4) * (3,)   → BŁĄD! {e}")
    print("  Rozwiązanie: reshape na (3,1)")

### Krotka `(3,)` vs `(3,1)` — klucz do broadcastingu

| Kształt | ndim | Co NumPy widzi | Mnożenie z `(3,4)` |
|---------|------|----------------|---------------------|
| `(3,)` | 1 | `[a, b, c]` — wektor 1D | Od prawej: **3 ≠ 4 → BŁĄD!** |
| `(3,1)` | 2 | `[[a], [b], [c]]` — kolumna 2D | Od prawej: **1→4, 3==3 → OK!** |

**Zapamiętaj:** `.reshape(-1, 1)` to nie formalność — to informacja dla NumPy: *rozciągaj w prawo, po kolumnach*.

In [ ]:
# Błąd i rozwiązanie w praktyce
mnoznik = np.array([1, 2, 3])
print(f"mnoznik.shape = {mnoznik.shape} — wektor 1D")

try:
    ceny * mnoznik
except ValueError:
    print("ceny (3,4) * mnoznik (3,) → BŁĄD!")

mnoznik_kol = mnoznik.reshape(-1, 1)  # (3,) → (3,1)
print(f"\nPo reshape(-1, 1): shape = {mnoznik_kol.shape}")
print(f"Wynik:\n{ceny * mnoznik_kol}")  # teraz działa!

> **Ciekawostka:** Broadcasting to jeden z powodów, dla których NumPy jest 10-100× szybszy
> od pętli w Pythonie. Pod spodem NumPy korzysta z instrukcji **SIMD** procesora
> (Single Instruction, Multiple Data) — tego samego mechanizmu,
> który przyspiesza grafikę w grach komputerowych.

---

## 2. Reshape i stacking — zmiana kształtu tablic

Dane nie zawsze przychodzą w formacie, który potrzebujemy. Reshape pozwala reorganizować dane.

| Metoda | Co robi | Kiedy używać |
|--------|---------|-------------|
| `a.reshape(w, k)` | Zmienia kształt | Dane płaskie → macierz |
| `a.reshape(w, -1)` | Auto-wymiar | NumPy oblicza brakujący wymiar |
| `a.flatten()` | Spłaszcza do 1D | Macierz → wektor (kopia) |
| `np.vstack([a, b])` | Pionowo | Nowe wiersze |
| `np.hstack([a, b])` | Poziomo | Dłuższy wektor |
| `np.column_stack([a, b])` | Kolumny | Każdy wektor jako kolumna |

In [ ]:
# Reshape — zmiana kształtu
a = np.arange(12)
print(f"Flat (12 elementów): {a}")
print(f"Reshape 3×4:\n{a.reshape(3, 4)}")
print(f"Reshape 4×3:\n{a.reshape(4, 3)}")
print(f"Reshape z -1 (auto): {a.reshape(3, -1).shape}  → NumPy sam oblicza 12/3=4")

In [ ]:
# Reshape — jak elementy wypełniają macierz
print(f"a = {a}")   # [0,1,2,...,11]
print()
print("reshape(3, 4) — elementy wypełniają WIERSZAMI:")
print("  wiersz 0: [0, 1, 2, 3]    — pierwsze 4 elementy")
print("  wiersz 1: [4, 5, 6, 7]    — następne 4")
print("  wiersz 2: [8, 9, 10, 11]  — ostatnie 4")
print()
print("reshape(4, 3) — INNE ułożenie (nie transpozycja!):")
print(a.reshape(4, 3))

In [ ]:
# Niedozwolony reshape — liczba elementów się nie zgadza
try:
    a.reshape(3, 5)   # 3*5=15 != 12
except ValueError as e:
    print(f"Błąd: {e}")
    print("  12 elementów nie da się ułożyć w 3x5 (potrzeba 15)")

In [ ]:
# Najważniejszy reshape: (N,) → (N,1) — wektor na kolumnę
# To właśnie ten reshape, który naprawia błędy broadcastingu!
wektor = np.array([100, 200, 300])
kolumna = wektor.reshape(-1, 1)
print(f"Wektor:  shape={wektor.shape}  → {wektor}")
print(f"Kolumna: shape={kolumna.shape} →")
print(kolumna)

### Reshape vs transpose — to nie to samo!

```
Oryginał (2×3):    reshape(3,2):    transpose (.T → 3×2):
[[1, 2, 3],        [[1, 2],         [[1, 4],
 [4, 5, 6]]         [3, 4],          [2, 5],
                    [5, 6]]          [3, 6]]
```

- **reshape** — elementy wypełniają wierszami (jak czytanie książki)
- **transpose (.T)** — zamienia wiersze z kolumnami (odbicie lustrzane)

In [ ]:
# Reshape vs transpose — porównanie
m = np.array([[1, 2, 3], [4, 5, 6]])
print(f"Oryginał (2×3):\n{m}\n")
print(f"reshape(3, 2):\n{m.reshape(3, 2)}\n")
print(f"transpose (.T):\n{m.T}")
# Różne wyniki! To różne operacje.

In [ ]:
# Flatten — spłaszczanie macierzy do 1D
macierz = np.array([[1, 2, 3], [4, 5, 6]])
print(f"Macierz:\n{macierz}")
print(f"Flatten: {macierz.flatten()}")  # kopia — bezpieczne

### Łączenie tablic (stacking)

In [ ]:
# vstack — pionowo (wiersze pod sobą)
q1 = np.array([100, 200, 150])
q2 = np.array([120, 210, 160])
q3 = np.array([130, 230, 180])

print(f"vstack (wiersze pod sobą):")
print(np.vstack([q1, q2, q3]))

In [ ]:
# hstack — poziomo (dłuższy wektor)
print(f"hstack (obok siebie): {np.hstack([q1, q2, q3])}")

In [ ]:
# column_stack — każda tablica jako kolumna
print(f"column_stack (kolumny obok siebie):")
print(np.column_stack([q1, q2, q3]))
# Przydatne gdy łączysz dane z różnych źródeł

---

## 3. Operacje warunkowe i sortowanie

### np.where() — warunkowe przypisanie

`np.where(warunek, wartość_True, wartość_False)` — jak `=JEŻELI()` w Excelu, ale na **całej tablicy** naraz.

In [ ]:
# np.where — zdał / nie zdał
oceny = np.array([3.0, 4.5, 2.0, 5.0, 3.5, 4.0, 2.5])
status = np.where(oceny >= 3.0, 'ZDAŁ', 'NIE ZDAŁ')
print(f"Oceny:  {oceny}")
print(f"Status: {status}")

In [ ]:
# np.where z imionami — pełny raport
imiona = ['Anna', 'Bartek', 'Celina', 'Dawid', 'Ewa', 'Filip', 'Grażyna']
premia = np.where(oceny >= 4.0, 500, 0)
print("Wyniki egzaminu:")
for imie, ocena, s, p in zip(imiona, oceny, status, premia):
    print(f"  {imie:8s}: {ocena} -> {s}, premia: {p} zł")

In [ ]:
# np.where z obliczeniami — różne stawki rabatu
ceny_prod = np.array([50, 150, 300, 80, 500, 1200])
ceny_po = np.where(ceny_prod > 200, ceny_prod * 0.85, ceny_prod * 0.95)
print(f"Ceny oryginalne: {ceny_prod}")
print(f"Po rabacie:      {np.round(ceny_po, 0).astype(int)}")
# Droższe niż 200 → rabat 15%, tańsze → rabat 5%

### np.select() — wielowarunkowe przypisanie

`np.where` obsługuje 2 warianty (tak/nie). A co gdy masz 3-5 kategorii?
Zamiast zagnieżdżać `np.where` — użyj `np.select`.

**Kluczowa zasada:** kolejność warunków ma znaczenie! Pierwszy spełniony wygrywa.

In [ ]:
# np.select — segmentacja klientów wg obrotu
obroty = np.array([15000, 3500, 800, 52000, 9200, 1100, 28000])

warunki = [obroty >= 20000, obroty >= 5000, obroty >= 1000]
wartosci = ['PREMIUM', 'STANDARD', 'BASIC']

segment = np.select(warunki, wartosci, default='MICRO')
print(f"Obroty:   {obroty}")
print(f"Segmenty: {segment}")

In [ ]:
# np.select — system ocen (jak zagnieżdżone IF w Excelu)
punkty = np.array([95, 72, 58, 83, 44, 91, 67])
ocena = np.select(
    [punkty >= 90, punkty >= 75, punkty >= 60, punkty >= 50],
    ['bardzo dobry', 'dobry', 'dostateczny', 'mierny'],
    default='niedostateczny'
)
print("System ocen:")
for p, o in zip(punkty, ocena):
    print(f"  {p} pkt -> {o}")

In [ ]:
# Co się stanie jak odwrócimy kolejność?
segment_zle = np.select(
    [obroty >= 1000, obroty >= 5000, obroty >= 20000],  # ZŁA kolejność!
    ['BASIC', 'STANDARD', 'PREMIUM'],
    default='MICRO'
)
print(f"Źle: {segment_zle}")
print("52000 >= 1000 → 'BASIC' (pierwszy spełniony warunek!)")
print("Dlatego ZAWSZE od najwyższego progu do najniższego.")

### np.clip() — przycinanie wartości do zakresu

Przydatne do walidacji (oceny 2-5), normalizacji (procenty 0-100), usuwania outlierów.

In [ ]:
# np.clip — przycinanie wartości do zakresu
oceny_brudne = np.array([1.5, 3.0, 5.5, 4.0, 0.0, 6.0])
oceny_czyste = np.clip(oceny_brudne, 2.0, 5.0)
print(f"Brudne:        {oceny_brudne}")
print(f"Po clip(2, 5): {oceny_czyste}")
# Wartości poza skalą 2-5 przycięte do granic

### Sortowanie i ranking

- **`np.sort(a)`** — posortowana **kopia** (oryginał bez zmian)
- **`np.argsort(a)`** — **indeksy** sortowania — klucz do rankingów!

```
dane:    [340, 120, 560, 90, 420]
sort:    [90, 120, 340, 420, 560]   ← wartości posortowane
argsort: [3, 1, 0, 4, 2]           ← INDEKSY: element [3]=90 jest najmniejszy
```

In [ ]:
# np.sort — posortowana kopia (oryginał bez zmian)
sprzedaz = np.array([340, 120, 560, 90, 420])
print(f"Oryginał:   {sprzedaz}")
print(f"Rosnąco:    {np.sort(sprzedaz)}")
print(f"Malejąco:   {np.sort(sprzedaz)[::-1]}")
print(f"Oryginał:   {sprzedaz}")  # nie zmieniony!

In [ ]:
# np.argsort — indeksy sortowania (do rankingów!)
produkty = ['A', 'B', 'C', 'D', 'E']
indeksy = np.argsort(sprzedaz)[::-1]  # indeksy od najlepszego

print("Ranking sprzedaży:")
for i, idx in enumerate(indeksy, 1):
    print(f"  {i}. {produkty[idx]} — {sprzedaz[idx]} szt.")

### np.unique() — unikalne wartości i zliczanie

In [ ]:
# np.unique — zliczanie kategorii
transakcje = np.array(['IT', 'HR', 'IT', 'Finanse', 'HR', 'IT', 'Finanse', 'Finanse'])
unikalne, ile = np.unique(transakcje, return_counts=True)
print(f"Działy:    {unikalne}")
print(f"Transakcje: {ile}")
# Jak value_counts() w Pandas — poznacie za tydzień

In [ ]:
# np.unique na liczbach — rozkład ocen
oceny_grupy = np.array([3, 5, 3, 4, 5, 5, 3, 4, 2, 4])
wartosci, liczby = np.unique(oceny_grupy, return_counts=True)
print("Rozkład ocen:")
for w, l in zip(wartosci, liczby):
    print(f"  Ocena {w}: {l} student(ów)")

### np.digitize() — przypisywanie do przedziałów (binning)

Przydatne do tworzenia grup wiekowych, klas dochodowych, przedziałów cenowych.

```
bins = [18, 30, 45, 60]
Przedziały:  <18  |  18-30  |  31-45  |  46-60  |  >60
Indeksy:      0   |    1    |    2    |    3    |    4
```

In [ ]:
# np.digitize — grupy wiekowe klientów
wiek = np.array([22, 35, 48, 19, 67, 55, 31, 42])
bins_wiek = [18, 30, 45, 60]
grupy = np.digitize(wiek, bins_wiek)
etykiety = ['<18', '18-30', '31-45', '46-60', '60+']

print("Segmentacja wiekowa:")
for w, g in zip(wiek, grupy):
    print(f"  Wiek {w} → {etykiety[g]}")

In [ ]:
# np.digitize — przedziały cenowe
ceny_prod = np.array([29, 150, 499, 75, 1200, 350, 15, 89])
bins_ceny = [50, 100, 500]
kategorie_idx = np.digitize(ceny_prod, bins_ceny)
kategorie = ['budżetowy', 'ekonomiczny', 'premium', 'luksusowy']

print("Kategorie cenowe:")
for c, ki in zip(ceny_prod, kategorie_idx):
    print(f"  {c} zł -> {kategorie[ki]}")

### Korelacja — np.corrcoef()

Wynik `r` (współczynnik Pearsona) jest w zakresie **-1 do 1**:

| Wartość r | Interpretacja | Przykład |
|-----------|---------------|----------|
| r ≈ 0.7–1.0 | Silna dodatnia | Więcej reklamy → większa sprzedaż |
| r ≈ 0 | Brak zależności | Kolor oczu → ocena z matmy |
| r ≈ -0.7 do -1.0 | Silna ujemna | Cena biletu → liczba pasażerów |

**Uwaga:** Korelacja to **nie** przyczynowość! `r = 0.85` oznacza, że zmienne zmieniają się razem, ale nie mówi, która powoduje drugą.

In [ ]:
# Korelacja: reklama vs sprzedaż
np.random.seed(42)
reklama = np.random.normal(1000, 200, 30)
sprzedaz_kor = 500 + 0.8 * reklama + np.random.normal(0, 100, 30)

r = np.corrcoef(reklama, sprzedaz_kor)[0, 1]
print(f"Korelacja reklama vs sprzedaż: r = {r:.3f}")
# r ~ 0.85 — silna dodatnia korelacja

In [ ]:
# Macierz korelacji — pełna
korelacja = np.corrcoef(reklama, sprzedaz_kor)
print(f"Macierz korelacji:\n{np.round(korelacja, 3)}")
# [0,0] i [1,1] = 1.0 (korelacja z samym sobą)
# [0,1] i [1,0] = r (korelacja między zmiennymi)

> **Sk\u0105d nazwa *wsp\u00f3\u0142czynnik Pearsona*?** Karl Pearson (1857-1936) to brytyjski matematyk,
> jeden z założycieli nowoczesnej statystyki. Stworzył też pojęcie odchylenia standardowego
> i chi-kwadrat. Jego wzór korelacji z 1895 roku jest używany do dziś — w NumPy, Excelu,
> SPSS i każdym innym narzędziu statystycznym.

---

## 4. Brakujące dane — np.nan

W realnych danych **zawsze** są braki: klient nie podał wieku, czujnik się zepsuł, pusta komórka w Excelu. NumPy oznacza braki jako `np.nan` (Not a Number).

**Problem:** jeden `nan` zaraża cały wynik!

| Zwykła funkcja | Wynik | Funkcja nan-safe | Wynik |
|----------------|-------|------------------|-------|
| `a.mean()` | `nan` | `np.nanmean(a)` | poprawna średnia |
| `a.sum()` | `nan` | `np.nansum(a)` | poprawna suma |
| `a.std()` | `nan` | `np.nanstd(a)` | poprawne std |

In [ ]:
# np.nan — brakujące dane
temperatura = np.array([21.5, 22.0, np.nan, 23.1, np.nan, 20.8, 22.5])
print(f"Temperatura: {temperatura}")
print(f"Zwykła średnia:  {temperatura.mean()}")       # nan!
print(f"nanmean:         {np.nanmean(temperatura):.1f}")  # poprawna
print(f"Ile NaN:         {np.isnan(temperatura).sum()}")
print(f"Gdzie NaN:       indeksy {np.where(np.isnan(temperatura))[0]}")

In [ ]:
# Realistyczny przykład — wynagrodzenia z brakami
wynagrodzenia_nan = np.array([4500, np.nan, 5200, 6100, np.nan, 3800, 5500, np.nan, 4900])
print(f"Dane: {wynagrodzenia_nan}")
print(f"Braki: {np.isnan(wynagrodzenia_nan).sum()} z {len(wynagrodzenia_nan)}")
print(f"Średnia (bez braków): {np.nanmean(wynagrodzenia_nan):.0f} zł")
print(f"Mediana (bez braków): {np.nanmedian(wynagrodzenia_nan):.0f} zł")
print(f"Min: {np.nanmin(wynagrodzenia_nan):.0f}, Max: {np.nanmax(wynagrodzenia_nan):.0f}")

In [ ]:
# 3 strategie radzenia sobie z brakami
dane = np.array([10.0, np.nan, 30.0, np.nan, 50.0])
print(f"Dane: {dane}\n")

# 1. Ignoruj (nanmean, nansum...)
print(f"1. Ignoruj: nanmean = {np.nanmean(dane):.1f}")

# 2. Zamień na średnią
dane_filled = dane.copy()
dane_filled[np.isnan(dane_filled)] = np.nanmean(dane)
print(f"2. Zamień:  {dane_filled}")

# 3. Usuń
dane_clean = dane[~np.isnan(dane)]
print(f"3. Usuń:    {dane_clean}")
print("\nNa zjeździe 5 poznacie dropna() i fillna() w Pandas — wygodniejsze.")

---

## 5. Generowanie danych i statystyki opisowe

Po co generować dane? **Testowanie** (nie masz prawdziwych danych), **symulacje** (modelujesz scenariusze), **nauka** (ćwiczysz na danych o znanych właściwościach).

| Rozkład | Funkcja | Przykład zastosowania |
|---------|---------|----------------------|
| Normalny (Gaussa) | `np.random.normal(mean, std, n)` | Wynagrodzenia, wzrost |
| Jednostajny | `np.random.uniform(low, high, n)` | Losowe ceny |
| Poissona | `np.random.poisson(lambda, n)` | Zamówienia/dzień |
| Losowy wybór | `np.random.choice(tablica, n)` | Losowanie z listy |

**`np.random.seed(42)`** — te same *losowe* liczby za każdym razem. Wyniki powtarzalne.

In [ ]:
# Rozkład normalny — wynagrodzenia (średnia 5000, odchylenie 1000)
np.random.seed(42)
wynagrodzenia = np.random.normal(loc=5000, scale=1000, size=100)
print(f"Wynagrodzenia (pierwszych 10): {wynagrodzenia[:10].astype(int)}")
print(f"Średnia: {wynagrodzenia.mean():.0f}, Std: {wynagrodzenia.std():.0f}")

In [ ]:
# Co oznaczają parametry rozkładu normalnego?
print("np.random.normal(loc=5000, scale=1000, size=100)")
print("  loc=5000   — średnia (centrum rozkładu)")
print("  scale=1000 — odchylenie standardowe (rozrzut)")
print("  size=100   — ile liczb wygenerować")
print()
print("Rozkład normalny (dzwonowy):")
print(f"  ~68% danych w [{5000-1000}, {5000+1000}]  (średnia +/- 1 std)")
print(f"  ~95% danych w [{5000-2000}, {5000+2000}]  (średnia +/- 2 std)")

In [ ]:
# Rozkład jednostajny — ceny w przedziale [10, 500]
ceny_losowe = np.random.uniform(low=10, high=500, size=50)
print(f"Ceny (pierwszych 8): {np.round(ceny_losowe[:8], 2)}")
print(f"Min: {ceny_losowe.min():.0f}, Max: {ceny_losowe.max():.0f}")
# Każda wartość z przedziału równie prawdopodobna

In [ ]:
# Rozkład Poissona — liczba zamówień dziennie (średnio 20)
zamowienia = np.random.poisson(lam=20, size=30)
print(f"Zamówienia dziennie (30 dni): {zamowienia}")
print(f"Średnia: {zamowienia.mean():.1f}, Min: {zamowienia.min()}, Max: {zamowienia.max()}")

### Statystyki opisowe — percentyle, IQR, mediana

**Kiedy średnia, kiedy mediana?**
- **Średnia** — dobra dla danych symetrycznych
- **Mediana** — odporna na outlier'y

Przykład: firma z 5 pracownikami. Zarobki: 3000, 3500, 4000, 4500, **50000** (prezes).
- Średnia = 13 000 zł — zawyżona przez prezesa!
- Mediana = 4 000 zł — lepiej oddaje *typowe* zarobki

In [ ]:
# Średnia vs mediana — efekt outliera
zarobki = np.array([3000, 3500, 4000, 4500, 50000])
print(f"Zarobki: {zarobki}")
print(f"Średnia: {zarobki.mean():.0f} zł  — zawyżona przez prezesa!")
print(f"Mediana: {np.median(zarobki):.0f} zł  — 'typowe' zarobki")

In [ ]:
# Pełny zestaw statystyk opisowych
dane = wynagrodzenia
print(f"Średnia:  {dane.mean():.0f}")
print(f"Mediana:  {np.median(dane):.0f}")
print(f"Std:      {dane.std():.0f}")
print(f"Min:      {dane.min():.0f}")
print(f"Max:      {dane.max():.0f}")

In [ ]:
# Percentyle i IQR
dane = wynagrodzenia
q1 = np.percentile(dane, 25)
q3 = np.percentile(dane, 75)
iqr = q3 - q1

print(f"Średnia:  {dane.mean():.0f}")
print(f"Mediana:  {np.median(dane):.0f}")
print(f"Q1 (25%): {q1:.0f}")
print(f"Q3 (75%): {q3:.0f}")
print(f"IQR:      {iqr:.0f}")
# IQR = rozstęp międzykwartylowy — miara rozproszenia odporna na outlier'y

### np.diff() i np.cumsum() — analiza trendów

```
dane:   [100, 150, 120, 200]
cumsum: [100, 250, 370, 570]   ← ile łącznie po każdym miesiącu
diff:   [50, -30, 80]          ← zmiana miesiąc do miesiąca (N-1 elementów!)
```

In [ ]:
# Suma kumulacyjna — narastające przychody
miesieczne = np.array([100, 150, 120, 200, 180, 160])
print(f"Przychody miesięczne: {miesieczne}")
print(f"Suma kumulacyjna:     {miesieczne.cumsum()}")
# [100, 250, 370, 570, 750, 910] — ile łącznie po każdym miesiącu

In [ ]:
# np.diff — dynamika sprzedaży miesiąc do miesiąca
sprzedaz_mies = np.array([100, 120, 115, 140, 135, 160])
zmiany = np.diff(sprzedaz_mies)
zmiana_proc = zmiany / sprzedaz_mies[:-1] * 100

print(f"Sprzedaż:   {sprzedaz_mies}")
print(f"Zmiany m/m: {zmiany}")
print(f"Zmiany %:   {np.round(zmiana_proc, 1)}")
print(f"Kumulacyjnie: {sprzedaz_mies.cumsum()}")

In [ ]:
# Zmiana procentowa — oddzielnie
zmiana_proc = np.diff(sprzedaz_mies) / sprzedaz_mies[:-1] * 100
print("Zmiany procentowe m/m:")
for i, z in enumerate(zmiana_proc):
    symbol = '+' if z > 0 else ''
    print(f"  Miesiąc {i+1} -> {i+2}: {symbol}{z:.1f}%")

---

## 6. Mini-ćwiczenia na wykładzie

### Ćwiczenie A — Broadcasting w sklepie

In [ ]:
# Dane: 4 produkty × 3 miesiące, ceny w złotych
ceny_sklep = np.array([[25, 28, 30],
                        [100, 95, 110],
                        [45, 50, 48],
                        [200, 210, 195]])
produkty_sklep = ['Kubek', 'Plecak', 'Koszulka', 'Buty']
print(f"Ceny (4 produkty × 3 miesiące):\n{ceny_sklep}")

In [ ]:
# 1. Podwyżka 10% (broadcasting skalarny)
print(f"Po podwyżce 10%:\n{np.round(ceny_sklep * 1.10, 2)}")

In [ ]:
# 2. Rabat sezonowy: Sty=0%, Lut=5%, Mar=15% (broadcasting wierszowy)
rabat_sezon = np.array([0.0, 0.05, 0.15])
print(f"Po rabacie:\n{np.round(ceny_sklep * (1 - rabat_sezon), 2)}")

In [ ]:
# 3. Marża per produkt (broadcasting kolumnowy — reshape!)
marza = np.array([1.50, 1.30, 1.40, 1.25]).reshape(-1, 1)
print(f"Z marżą:\n{np.round(ceny_sklep * marza, 2)}")

In [ ]:
# Sprawdzenie: dlaczego reshape(-1, 1) był potrzebny?
print("Bez reshape:")
print(f"  ceny.shape:  (4, 3)")
print(f"  marza.shape: {np.array([1.50, 1.30, 1.40, 1.25]).shape}  — (4,)")
print(f"  Od prawej: 3 != 4 — BŁĄD!")
print()
print("Z reshape(-1, 1):")
print(f"  marza.shape: {np.array([1.50, 1.30, 1.40, 1.25]).reshape(-1,1).shape} — (4, 1)")
print(f"  Od prawej: 1->3, 4==4 — OK!")

### Ćwiczenie B — Reshape i analiza kwartalnych wyników

In [ ]:
# Dane płaskie — 12 miesięcy sprzedaży
sprzedaz_flat = np.array([120, 135, 110, 145, 160, 155, 170, 180, 165, 190, 200, 210])

# Reshape na kwartały
kwartaly = sprzedaz_flat.reshape(4, 3)
sumy = kwartaly.sum(axis=1)

for i, s in enumerate(sumy, 1):
    print(f"  Q{i}: {s} szt.")
print(f"Najlepszy: Q{sumy.argmax() + 1} ({sumy.max()} szt.)")

### Ćwiczenie C — Analiza danych finansowych

In [ ]:
# Firma z 5 oddziałami — przychody miesięczne (tys. zł)
np.random.seed(123)
oddzialy = ['Warszawa', 'Kraków', 'Wrocław', 'Gdańsk', 'Poznań']
przychody = np.random.randint(50, 200, size=(5, 12))

# 1. Najwyższy roczny przychód
roczne = przychody.sum(axis=1)
print(f"Najwyższy: {oddzialy[roczne.argmax()]} ({roczne.max()} tys. zł)")

# 2. Korelacja Warszawa-Kraków
r = np.corrcoef(przychody[0], przychody[1])[0, 1]
print(f"Korelacja Warszawa-Kraków: r = {r:.3f}")

# 3. IQR Warszawy
waw = przychody[0]
print(f"Warszawa — Q1: {np.percentile(waw, 25):.0f}, Med: {np.median(waw):.0f}, Q3: {np.percentile(waw, 75):.0f}")

---

## Podgląd laboratorium

Na laboratorium przećwiczycie to wszystko na danych biznesowych:

**Start — 3 komendy:**
```
cd C:\Users\student\python2
.venv\Scripts\Activate.ps1
code .
```

**Ćwiczenie 1:** Broadcasting w sklepie — ceny 4 produktów w 3 sklepach, inflacja, rabaty per sklep, marża per produkt

**Ćwiczenie 2:** Reshape — dane płaskie → macierz kwartałów, łączenie danych z oddziałów (vstack)

**Ćwiczenie 3:** Samodzielna analiza finansowa — wynagrodzenia, premie (np.where), ranking (argsort), korelacja

**Ćwiczenie 4:** Generowanie 100 klientów — rozkład normalny, Poisson, segmentacja

**Ćwiczenie 5:** Commit na GitHub

---

## Na koniec

> *Nie życz sobie, żeby było łatwiej. Życz sobie, żebyś był lepszy.*
> — Jim Rohn, *The Treasury of Quotes*

**Po dzisiejszym wykładzie kończymy z czystym NumPy.** Od W05 — Pandas. NumPy robił obliczenia — Pandas doda etykiety, nazwy kolumn i wygodne metody.

**Zadanie domowe (nieoceniane):** Wygenerujcie dane o 50 pracownikach: pensja (normal, średnia 6000, std 1500), staż (randint 1-20), ocena (uniform 1-5). Policzcie korelację staż-pensja. Wrzućcie notebook na GitHub.

## Podsumowanie — ściąga

| Funkcja | Co robi | Przykład |
|---------|---------|----------|
| `reshape(n, m)` | Zmienia kształt | `a.reshape(3, 4)` |
| `reshape(-1, 1)` | Wektor → kolumna | `v.reshape(-1, 1)` |
| `.T` | Transpozycja | `macierz.T` |
| `flatten()` | Spłaszczenie do 1D | `m.flatten()` |
| `vstack/hstack` | Łączenie tablic | `np.vstack([a, b])` |
| `np.where` | If/else na tablicy | `np.where(x > 0, 'tak', 'nie')` |
| `np.select` | Wiele warunków | `np.select([w1, w2], [v1, v2])` |
| `np.clip` | Przycinanie zakresu | `np.clip(x, 0, 100)` |
| `np.sort` | Sortowanie (kopia) | `np.sort(x)` |
| `np.argsort` | Indeksy sortowania | `np.argsort(x)[::-1]` |
| `np.unique` | Unikalne + zliczanie | `np.unique(x, return_counts=True)` |
| `np.digitize` | Binning | `np.digitize(x, [10, 20, 30])` |
| `np.corrcoef` | Korelacja Pearsona | `np.corrcoef(x, y)[0,1]` |
| `np.nan` | Brak danych | `np.nanmean(x)` |
| `np.random.*` | Generowanie danych | `np.random.normal(0, 1, 100)` |
| `np.cumsum` | Suma kumulacyjna | `x.cumsum()` |
| `np.diff` | Różnice między elementami | `np.diff(x)` |

---

## Źródła

| Źródło | Opis |
|--------|------|
| [NumPy Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html) | Oficjalna dokumentacja |
| [NumPy Array Manipulation](https://numpy.org/doc/stable/reference/routines.array-manipulation.html) | reshape, flatten, stacking |
| [NumPy Sorting](https://numpy.org/doc/stable/reference/routines.sort.html) | sort, argsort, unique |
| [NumPy Random](https://numpy.org/doc/stable/reference/random/) | Generowanie danych |
| [NumPy Statistics](https://numpy.org/doc/stable/reference/routines.statistics.html) | Statystyki opisowe |
| [Real Python — Broadcasting](https://realpython.com/numpy-array-programming/) | Praktyczny poradnik |